# 09 — LangGraph Fundamentals: State, Nodes, Edges

## Learning requirements
- hiểu LangGraph là low-level orchestration/runtime, không phải “LangChain nâng cấp”;
- define explicit state;
- node = state transition;
- edge = control flow;
- conditional routing;
- reducers;
- biết khi nào graph deterministic tốt hơn agent loop.

Một graph không cần mọi node đều gọi LLM.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class CandidateState(TypedDict):
    name: str
    years: float
    score: int
    decision: str

def validate(state: CandidateState):
    if state["years"] < 0:
        raise ValueError("years cannot be negative")
    return {}

def score(state: CandidateState):
    # Deterministic demo logic.
    score_value = min(100, int(state["years"] * 20))
    return {"score": score_value}

def route(state: CandidateState) -> Literal["accept", "reject"]:
    return "accept" if state["score"] >= 60 else "reject"

def accept(state: CandidateState):
    return {"decision": "accept"}

def reject(state: CandidateState):
    return {"decision": "reject"}

builder = StateGraph(CandidateState)
builder.add_node("validate", validate)
builder.add_node("score", score)
builder.add_node("accept", accept)
builder.add_node("reject", reject)

builder.add_edge(START, "validate")
builder.add_edge("validate", "score")
builder.add_conditional_edges("score", route, {
    "accept": "accept",
    "reject": "reject",
})
builder.add_edge("accept", END)
builder.add_edge("reject", END)

graph = builder.compile()
print(graph.invoke({"name": "A", "years": 3.5, "score": 0, "decision": ""}))

## State design rules

State không nên là “dump mọi object vào một dict”.

Hãy hỏi:
- field nào là durable business state?
- field nào chỉ là derived temporary data?
- field nào chứa large payload nên lưu reference/path thay vì full content?
- field nào cần reducer khi parallel branches cùng update?

## Graph API vs Functional API

Học cả hai:
- **Graph API**: explicit nodes/edges, visualization, branching rõ.
- **Functional API**: procedural Python quen thuộc nhưng vẫn tận dụng persistence/durable execution.

Bài tập: implement cùng workflow candidate scoring bằng Functional API rồi ghi comparison.

## Required output
1. Graph candidate workflow.
2. Thêm nhánh `manual_review` khi score nằm 50–69.
3. Vẽ state transition.
4. Viết comparison Graph API vs Functional API.

## Done criteria
Bạn có thể chỉ ra node nào nên deterministic và node nào đáng dùng LLM.